In [22]:
import pandas as pd
import pyarrow.parquet as pq
import sys
import os
import numpy as np
import random as r
import json 

sys.path.insert(0, os.path.join(os.getcwd(), 'chronoepilogi_implementation'))

# Now import the class
from ce_extensions2 import ChronoEpilogi

In [14]:
###ONE HOT ENCODING OF THE CLASS

db = pd.read_parquet("RUL_FRMTM.parquet", engine="fastparquet")
##Cleaning of all the row having a nan for a columns 
##One-Hot encoding 
column_to_OHE="Class"
db_encoded = pd.get_dummies(db, prefix="Class_", columns=[column_to_OHE], dtype=int)

#########################################################
# THIS CAN BE USED TO REDUCE THE DATASET SIZE IF NEEDED
non_na_rows = db[~db["System"].isna()]
na_rows = db[db["System"].isna()].sample(n=3000, random_state=42)  # Adjust 'n' as needed
db_sampled = pd.concat([non_na_rows, na_rows])
db_sampled.sort_values(by=["Date_Time", "EPISODE_ID"], inplace=True)
db=db_sampled.copy()
##########################################################

db_encoded = pd.get_dummies(db_sampled, prefix="Class_", columns=[column_to_OHE])
print(f"Size of the sampled database: {db_encoded.shape}")

##Dropping of EPISODE_ID and DateTime to not let them appear in the Markov Boundaries
db_encoded = db_encoded.drop(columns="EPISODE_ID")

target = 'RUL'  #target column
target_type =  "continuous"
column_names = db_encoded.columns.tolist()


## we take all the columns wanted for the model
numerical_columns = [col for col in column_names if db_encoded[col].dtype == np.float64 and col != target]
categorical_columns = [col for col in column_names if col.startswith("Class_")]

## and add them to our features database
features_db = numerical_columns + categorical_columns + [target]


#X = pd.concat([features_db, db_encoded[target]], axis=1)
X = db_encoded[features_db].copy()

## To prevent dtype issue, we set the type to a suitable one

X[numerical_columns] = X[numerical_columns].astype(np.float64)
X[categorical_columns] = X[categorical_columns].astype(np.int64)

## Cleaning the NaN and the +- inf
for col in numerical_columns:
    X[col] = X[col].fillna(X[col].mean())

## Data summary
print(f"Keeping {len(numerical_columns)+len(categorical_columns)+len(target)} columns:")
print(f"Categorical columns: {len(categorical_columns)}")
print(f"Numerical columns: {len(numerical_columns)}")
print(f"Target:{target}")


variable_types = {
    **{col: "numerical" for col in numerical_columns},
    **{col: "categorical" for col in categorical_columns}
}


Size of the sampled database: (4740, 99)
Keeping 95 columns:
Categorical columns: 6
Numerical columns: 86
Target:RUL


In [23]:
fs_instance = ChronoEpilogi(
    X,
    target,
    phases="FBEV",
    target_type=target_type,
    start_with_univariate_autoregressive_model=False,
    default_max_lag=1,  
    variable_types=variable_types
)

fs_instance.fit()

print("selected set:", fs_instance.selected_set)
print("length of selected set:",len(fs_instance.selected_set),"and number of covariates in the dataset:",len(X.columns) -1)
print("number of Markov Boundaries:",fs_instance.get_total_number_sets())
print(fs_instance.equivalent_variables)

data = {
    "selected set": fs_instance.selected_set,
    "length of selected set" :len(fs_instance.selected_set),
    "and number of covariates in the dataset" :len(X.columns) -1,
    "number of Markov Boundaries":fs_instance.get_total_number_sets(),
    "features inside MB": fs_instance.equivalent_variables,
}

json_str = json.dumps(data, indent=5)
with open("ClassOHE_lag1.json", "w") as f:
    f.write(json_str)



selected set: ['NIS_Steady_State_Nitrogen_Pressure_bar', 'ES_PB_Total_Injection_Temperature_celcius', 'LP_M2_Pressure_bar', 'CS_C2_TIT400_celcius', 'DS_D2_Targeted_Pressure_bar', 'HP_M1_Pressure_bar', 'Class__Other/Process/Software', 'DS_D2_Nozzle_Pressure_bar', 'LP_M1_Pressure_bar', 'DS_D1_Fuel_Coupling_Pressure_bar', 'DS_D2_Fuel_Coupling_Temperature_celcius', 'LP_M1_Outflow_Temperature_celcius', 'DS_D2_Inlet_Pressure_bar', 'DS_D1_Nozzle_Temperature_celcius', 'DS_D1_Nozzle_Pressure_bar', 'CS_C1_PT10_bar', 'AM_Temperature_celcius', 'ES_PA_VS_V386_binary', 'Class__Pressure Alerts', 'CS_C1_TIT10_celcius', 'ES_PA_Total_Injection_Pressure_bar', 'CS_C1_TIT400_celcius', 'NIS_After_Operation_Nitrogen_Pressure_bar', 'CS_C2_TIT60_celcius', 'Class__Cooling System Faults', 'DS_D1_Fuel_Coupling_Temperature_celcius', 'DS_D1_Targeted_Pressure_bar', 'HP_M3_Pressure_bar', 'ES_PB_Pressure_bar', 'CS_C1_TIT60_celcius', 'CS_C2_PT10_bar', 'HP_M2_Temperature_celcius', 'LP_VS_V023_binary', 'DS_D2_Nozzle_Temp

In [ ]:
fs_instance = ChronoEpilogi(
    X,
    target,
    phases="FBEV",
    target_type=target_type,
    start_with_univariate_autoregressive_model=False,
    default_max_lag=6,  
    variable_types=variable_types
)

fs_instance.fit()

print("selected set:", fs_instance.selected_set)
print("length of selected set:",len(fs_instance.selected_set),"and number of covariates in the dataset:",len(X.columns) -1)
print("number of Markov Boundaries:",fs_instance.get_total_number_sets())
print(fs_instance.equivalent_variables)

data = {
    "selected set": fs_instance.selected_set,
    "length of selected set" :len(fs_instance.selected_set),
    "and number of covariates in the dataset" :len(X.columns) -1,
    "number of Markov Boundaries":fs_instance.get_total_number_sets(),
    "features inside MB": fs_instance.equivalent_variables,
}

json_str = json.dumps(data, indent=5)
with open("ClassOHE_lag6.json", "w") as f:
    f.write(json_str)

selected set: ['NIS_Steady_State_Nitrogen_Pressure_bar', 'ES_PB_Total_Injection_Temperature_celcius', 'LP_M2_Pressure_bar', 'CS_C2_TIT400_celcius', 'DS_D2_Targeted_Pressure_bar', 'HP_M1_Pressure_bar', 'Class__Other/Process/Software', 'ES_PB_Total_Injection_Pressure_bar', 'LP_M1_Pressure_bar', 'DS_D2_Fuel_Coupling_Temperature_celcius', 'DS_D2_Fuel_Coupling_Pressure_bar', 'ES_PB_Pressure_bar', 'CS_C1_PT10_bar', 'DS_D1_Fuel_Coupling_Pressure_bar', 'AM_Temperature_celcius', 'LP_M1_Outflow_Temperature_celcius', 'DS_D2_Nozzle_Pressure_bar', 'DS_D1_Nozzle_Pressure_bar', 'DS_D2_Inlet_Pressure_bar', 'Class__Pressure Alerts', 'CS_C2_TIT60_celcius', 'Class__Cooling System Faults', 'HP_M3_Pressure_bar', 'CS_C1_TIT10_celcius', 'CS_C1_TIT400_celcius', 'NIS_After_Operation_Nitrogen_Pressure_bar', 'HP_IVS_V038_binary', 'DS_D1_Fuel_Coupling_Temperature_celcius', 'DS_D1_Targeted_Pressure_bar', 'ES_PA_Total_Injection_Pressure_bar', 'LP_M1_VS_V027_binary', 'ES_PA_VS_V006_binary', 'DS_D2_Nozzle_Temperature

In [ ]:
fs_instance = ChronoEpilogi(
    X,
    target,
    phases="FBEV",
    target_type=target_type,
    start_with_univariate_autoregressive_model=False,
    default_max_lag=20,  
    variable_types=variable_types
)

fs_instance.fit()

print("selected set:", fs_instance.selected_set)
print("length of selected set:",len(fs_instance.selected_set),"and number of covariates in the dataset:",len(X.columns) -1)
print("number of Markov Boundaries:",fs_instance.get_total_number_sets())
print(fs_instance.equivalent_variables)

data = {
    "selected set": fs_instance.selected_set,
    "length of selected set" :len(fs_instance.selected_set),
    "and number of covariates in the dataset" :len(X.columns) -1,
    "number of Markov Boundaries":fs_instance.get_total_number_sets(),
    "features inside MB": fs_instance.equivalent_variables,
}

json_str = json.dumps(data, indent=5)
with open("ClassOHE_lag20.json", "w") as f:
    f.write(json_str)

selected set: ['NIS_Steady_State_Nitrogen_Pressure_bar', 'ES_PB_Total_Injection_Temperature_celcius', 'LP_M2_Pressure_bar', 'DS_D2_Targeted_Pressure_bar', 'CS_C2_TIT400_celcius', 'HP_M1_Pressure_bar', 'DS_D2_Requested_Pressure_bar', 'DS_D2_Fuel_Coupling_Temperature_celcius', 'ES_PB_Pressure_bar', 'LP_M1_Pressure_bar', 'ES_PB_Total_Injection_Pressure_bar', 'Class__Other/Process/Software', 'DS_D1_Fuel_Coupling_Pressure_bar', 'CS_C1_PT10_bar', 'AM_Temperature_celcius', 'CS_C2_TIT60_celcius', 'DS_D2_Nozzle_Pressure_bar', 'CS_C1_PT400_bar', 'DS_D2_Nozzle_Temperature_celcius', 'Class__Pressure Alerts', 'DS_D2_Inlet_Pressure_bar', 'DS_D1_Fuel_Coupling_Temperature_celcius', 'DS_D1_Nozzle_Temperature_celcius', 'DS_D1_Nozzle_Pressure_bar', 'LP_M1_VS_V616_binary', 'NIS_After_Operation_Nitrogen_Pressure_bar']
length of selected set: 26 and number of covariates in the dataset: 92
number of Markov Boundaries: 1
{'NIS_Steady_State_Nitrogen_Pressure_bar': ['NIS_Steady_State_Nitrogen_Pressure_bar'], 'E

In [19]:
class_features = [feat for feat in fs_instance.selected_set if feat.startswith("Class_")]
print(class_features)

['Class__Other/Process/Software', 'Class__Pressure Alerts']
